# TCC — Reprodução das análises a partir do GitHub

Este notebook usa caminhos relativos ao repositório, sem depender do Google Drive.

## 0. Instalação de bibliotecas

No Google Colab, rode a célula abaixo. No GitHub, ela serve como documentação das dependências.

In [ ]:
!pip install pandas numpy openpyxl pyarrow matplotlib statsmodels scikit-learn rapidfuzz tqdm -q

## 1. Configuração de caminhos

A estrutura esperada é: `apendices/planilhas`, `apendices/parquet`, `notebooks`, `src` e `resultados`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Como este notebook fica dentro da pasta notebooks/, usamos ".." para voltar à raiz do repo.
ROOT_DIR = Path("..").resolve()

PLANILHAS_DIR = ROOT_DIR / "apendices" / "planilhas"
PARQUET_DIR = ROOT_DIR / "apendices" / "parquet"
RESULTADOS_DIR = ROOT_DIR / "resultados"
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", ROOT_DIR)
print("Planilhas:", PLANILHAS_DIR)
print("Parquets:", PARQUET_DIR)
print("Resultados:", RESULTADOS_DIR)


## 2. Conferência de arquivos obrigatórios

Se algum aparecer como `FALTA`, suba o arquivo para a pasta indicada ou ajuste o nome no código.

In [ ]:
arquivos = {
    "Base ocupacional final com fuzzy": PLANILHAS_DIR / "cbo_final_com_felten_gmyrek_cod_FINAL_com_fuzzy.xlsx",
    "Tabela fuzzy Felten": PLANILHAS_DIR / "cbo_final_fuzzy_felten.xlsx",
    "Tabela embeddings classificados": PLANILHAS_DIR / "embedding_residual_openai_review_classificado.xlsx",
    "PNAD analítica": PARQUET_DIR / "pnad_2020_2025_analitica_ia.parquet",
    "PNAD Felten": PARQUET_DIR / "pnad_2020_2025_felten.parquet",
    "PNAD Gmyrek": PARQUET_DIR / "pnad_2020_2025_gmyrek.parquet",
    "Agregado Felten": PARQUET_DIR / "agregado_felten_cod_ano.parquet",
    "Agregado Gmyrek": PARQUET_DIR / "agregado_gmyrek_cod_ano.parquet",
}

faltantes = []

for nome, caminho in arquivos.items():
    if caminho.exists():
        print(f"OK    {nome}: {caminho.name}")
    else:
        print(f"FALTA {nome}: {caminho}")
        faltantes.append((nome, caminho))

print("\nTotal faltante:", len(faltantes))


## 3. Carregamento das bases finais

Esta etapa parte das bases já tratadas e não reprocessa os microdados brutos da PNAD.

In [ ]:
base_ocupacional = pd.read_excel(arquivos["Base ocupacional final com fuzzy"])
pnad = pd.read_parquet(arquivos["PNAD analítica"])
pnad_felten = pd.read_parquet(arquivos["PNAD Felten"])
pnad_gmyrek = pd.read_parquet(arquivos["PNAD Gmyrek"])
ag_felten = pd.read_parquet(arquivos["Agregado Felten"])
ag_gmyrek = pd.read_parquet(arquivos["Agregado Gmyrek"])

print("Base ocupacional:", base_ocupacional.shape)
print("PNAD analítica:", pnad.shape)
print("PNAD Felten:", pnad_felten.shape)
print("PNAD Gmyrek:", pnad_gmyrek.shape)
print("Agregado Felten:", ag_felten.shape)
print("Agregado Gmyrek:", ag_gmyrek.shape)


## 4. Cobertura dos indicadores na base ocupacional

In [ ]:
cols = ["AIOE", "Exposure", "Mean", "SD", "COD_Grupo_Base"]
for c in cols:
    if c in base_ocupacional.columns:
        cobertura = base_ocupacional[c].notna().mean() * 100
        print(f"{c}: {cobertura:.2f}% preenchido")
    else:
        print(f"{c}: coluna não encontrada")


## 5. Matriz de correlação Felten × Gmyrek

Esta é uma das evidências de convergência metodológica entre os indicadores.

In [ ]:
corr_cols = [c for c in ["AIOE", "Mean", "SD"] if c in base_ocupacional.columns]
corr = base_ocupacional[corr_cols].apply(pd.to_numeric, errors="coerce").corr()
display(corr)

plt.figure(figsize=(6,5))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns)
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar(label="Correlação de Pearson")
plt.title("Matriz de correlação entre indicadores de exposição à IA")

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        plt.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center")

plt.tight_layout()
plt.savefig(RESULTADOS_DIR / "matriz_correlacao_indicadores.png", dpi=300)
plt.show()


## 6. Distribuição dos indicadores

In [ ]:
if "AIOE" in base_ocupacional.columns:
    plt.figure(figsize=(8,5))
    plt.hist(pd.to_numeric(base_ocupacional["AIOE"], errors="coerce").dropna(), bins=30)
    plt.xlabel("AIOE")
    plt.ylabel("Número de ocupações")
    plt.title("Distribuição do índice AIOE")
    plt.tight_layout()
    plt.savefig(RESULTADOS_DIR / "distribuicao_aioe.png", dpi=300)
    plt.show()

if "Mean" in base_ocupacional.columns:
    plt.figure(figsize=(8,5))
    plt.hist(pd.to_numeric(base_ocupacional["Mean"], errors="coerce").dropna(), bins=30)
    plt.xlabel("Mean — Gmyrek")
    plt.ylabel("Número de ocupações")
    plt.title("Distribuição do indicador Mean")
    plt.tight_layout()
    plt.savefig(RESULTADOS_DIR / "distribuicao_gmyrek_mean.png", dpi=300)
    plt.show()


## 7. Resumo fuzzy matching

In [ ]:
fuzzy = pd.read_excel(arquivos["Tabela fuzzy Felten"])

score_cols = [c for c in fuzzy.columns if "fuzzy" in c.lower() and "score" in c.lower()]
print("Colunas candidatas de score fuzzy:", score_cols)

if score_cols:
    col_score = score_cols[0]
    fuzzy[col_score] = pd.to_numeric(fuzzy[col_score], errors="coerce")
    fuzzy_aplicado = fuzzy[fuzzy[col_score].notna()].copy()

    print("Linhas totais:", len(fuzzy))
    print("Linhas com fuzzy aplicado:", len(fuzzy_aplicado))
    print(fuzzy_aplicado[col_score].describe())

    plt.figure(figsize=(8,5))
    plt.hist(fuzzy_aplicado[col_score], bins=25)
    plt.xlabel("Fuzzy score")
    plt.ylabel("Frequência")
    plt.title("Distribuição dos scores fuzzy — casos aplicados")
    plt.tight_layout()
    plt.savefig(RESULTADOS_DIR / "distribuicao_fuzzy_score.png", dpi=300)
    plt.show()
else:
    print("Nenhuma coluna de fuzzy score encontrada.")


## 8. Resumo embeddings residuais

Observação: embeddings foram usados como análise residual/revisão e não incorporados automaticamente à base final.

In [ ]:
embedding = pd.read_excel(arquivos["Tabela embeddings classificados"])
print("Embedding:", embedding.shape)
print(embedding.columns.tolist())

sim_cols = [c for c in embedding.columns if "similarity" in c.lower() or "cosine" in c.lower()]
print("Colunas candidatas de similaridade:", sim_cols)

if sim_cols:
    col_sim = sim_cols[0]
    embedding[col_sim] = pd.to_numeric(embedding[col_sim], errors="coerce")
    print(embedding[col_sim].describe())

    plt.figure(figsize=(8,5))
    plt.hist(embedding[col_sim].dropna(), bins=30)
    plt.xlabel("Similaridade de cosseno")
    plt.ylabel("Frequência")
    plt.title("Distribuição da similaridade dos embeddings residuais")
    plt.tight_layout()
    plt.savefig(RESULTADOS_DIR / "distribuicao_embeddings_similarity.png", dpi=300)
    plt.show()
else:
    print("Nenhuma coluna de similaridade encontrada.")


## 9. Exemplo de análise agregada: exposição × renda

In [ ]:
for df_ag, score_col, nome in [
    (ag_felten, "AIOE", "Felten/AIOE"),
    (ag_gmyrek, "Mean", "Gmyrek/Mean"),
]:
    if score_col in df_ag.columns and "log_renda_media_ponderada" in df_ag.columns:
        temp = df_ag.copy()
        temp[score_col] = pd.to_numeric(temp[score_col], errors="coerce")
        temp["log_renda_media_ponderada"] = pd.to_numeric(temp["log_renda_media_ponderada"], errors="coerce")
        temp = temp.dropna(subset=[score_col, "log_renda_media_ponderada"])

        plt.figure(figsize=(8,5))
        plt.scatter(temp[score_col], temp["log_renda_media_ponderada"], alpha=0.6)
        plt.xlabel(nome)
        plt.ylabel("Log da renda média ponderada")
        plt.title(f"Exposição à IA × renda — {nome}")
        plt.tight_layout()
        plt.savefig(RESULTADOS_DIR / f"scatter_exposicao_renda_{score_col}.png", dpi=300)
        plt.show()
    else:
        print(f"Colunas necessárias não encontradas para {nome}")
